In [9]:
model_id = "YuWangX/mplus-8b"

In [10]:
import os

os.environ.get('HF_ENDPOINT')

'https://hf-mirror.com'

In [11]:
import torch

### MPlus

In [1]:
import torch
from transformers import AutoTokenizer
from modeling_mplus import MPlus

model = MPlus.from_pretrained("YuWangX/mplus-8b", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("YuWangX/mplus-8b")
model = model.to(torch.bfloat16) # need to call it again to cast the `inv_freq` in rotary_emb to bfloat16 as well
# model.put_ltm_to_numpy() 

/home/ccm/miniconda3/envs/memoryllm/lib/python3.12/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/ccm/miniconda3/envs/memoryllm/lib/python3.12/site-packages/ipykerne

Memory Pool Parameters: 1.3422 B


Loading checkpoint shards: 100%|██████████| 8/8 [00:02<00:00,  3.16it/s]


### MemoryLLM

In [4]:
from modeling_memoryllm import MemoryLLM
from configuration_memoryllm import MemoryLLMConfig
from transformers import AutoTokenizer
# config = MemoryLLMConfig.from_pretrained(model_id)
model = MemoryLLM.from_pretrained(model_id, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/home/ccm/miniconda3/envs/memoryllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - I

Memory Pool Parameters: 1.6777 B


Loading checkpoint shards: 100%|██████████| 8/8 [00:02<00:00,  3.34it/s]


In [18]:
model.initialized = torch.tensor(0, dtype=torch.uint8)

In [13]:
print(model.initialized)

1


In [2]:
device = "cuda:1"

In [3]:
model = model.to(device)

In [4]:

# Self-Update with the new context
ctx = "Last week, John had a wonderful picnic with David. During their conversation, David mentioned multiple times that he likes eating apples. Though he didn't mention any other fruits, John says he can infer that David also like bananas."

# please make sure the context to inject into the memory is larger than 16 tokens, this is the hard minimum when training the model. The memory will be disturbed when less than 16 tokens are injected into the memory. 
model.inject_memory(tokenizer(ctx, return_tensors='pt', add_special_tokens=False).input_ids.to(device), update_memory=True)

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


ValueError: not enough values to unpack (expected 5, got 3)

### memoryllm-8b-chat

In [20]:
messages = [{
    'role': 'user', "content": "What fruits does David like?",
}]

input_prompt = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, tokenize=False) # remove bos tokens as the model has its own trained bos embeddings.
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

inputs = tokenizer(input_prompt, return_tensors='pt').to(device)

outputs = model.generate(**inputs,
                         temperature=0.1,
                         max_new_tokens=20,
                         eos_token_id=terminators)

response = tokenizer.decode(outputs[0])
print(response)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


<|begin_of_text|><|begin_of_text|><|start_header_id|>user<|end_header_id|>

What fruits does David like?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

John: I'm not sure, but I think he likes all fruits. He's a fruit lover


### memoryllm-8b

In [8]:
# memoryllm-8b
inputs = tokenizer("Question: What fruits does David like? Answer: ", return_tensors='pt', add_special_tokens=False)
inputs = inputs.to(device)

outputs = model.generate(**inputs, max_new_tokens=20, temperature=0.1)

input_len = inputs.input_ids.shape[1]   # 或者 inputs['input_ids'].shape[1]

response = tokenizer.decode(outputs[0][input_len:])
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


1. Apples 2. Oranges 3. Bananas 4. Grapes 
